# MultiIndex jako dane wielowymiarowe

Ten notebook jest lokalnym, samodzielnym dodatkiem do zajec. Pokazuje, jak traktowac `MultiIndex` jako sposob zapisu danych 3D i 4D bez kopiowania zewnetrznych notebookow.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

## Dane 3D: miasto, dzien, godzina

Jedna kolumna wartosci plus indeks z trzema poziomami odpowiada tablicy 3D, ale lepiej znosi braki danych i pozwala wygodnie robic przekroje.

In [ ]:
districts = ["Centrum", "Mokotow", "Praga"]
days = pd.date_range("2026-03-02", periods=4, freq="D")
hours = [8, 12, 16, 20]

idx = pd.MultiIndex.from_product(
    [districts, days, hours],
    names=["district", "day", "hour"],
)

traffic = pd.Series(
    rng.poisson(lam=120, size=len(idx)),
    index=idx,
    name="rides",
)
traffic.head(10)

Przekroj po jednym poziomie zostawia pozostale wymiary w indeksie.

In [ ]:
traffic.xs("Mokotow", level="district").head(8)

In [ ]:
traffic.xs(16, level="hour").unstack("district")

## Braki danych i pelna siatka

W praktyce czesto nie mamy wszystkich kombinacji. Najpierw robimy kompletna siatke indeksu, a potem `reindex` pokazuje, gdzie brakuje obserwacji.

In [ ]:
sample = traffic.sample(frac=0.8, random_state=0).sort_index()
completed = sample.reindex(idx)

missing_by_district = completed.isna().groupby(level="district").sum()
missing_by_district

## Dane 4D: miasto, dzien, godzina, kanal

Czwarty poziom indeksu pozwala zapisac dodatkowy wymiar bez tworzenia wielu osobnych tabel.

In [ ]:
channels = ["app", "phone"]
idx4 = pd.MultiIndex.from_product(
    [districts, days, hours, channels],
    names=["district", "day", "hour", "channel"],
)

rides4 = pd.Series(
    rng.poisson(lam=70, size=len(idx4)),
    index=idx4,
    name="rides",
)
rides4.head(12)

In [ ]:
daily_channel = rides4.groupby(level=["day", "channel"]).sum().unstack("channel")
daily_channel

In [ ]:
rush_hours = rides4.loc[pd.IndexSlice[:, :, [8, 16], :]]
rush_hours.groupby(level=["district", "channel"]).mean().unstack("channel")

## Mini-zadania

1. Policz laczna liczbe przejazdow dla kazdej dzielnicy i kanalu.
2. Wybierz tylko obserwacje weekendowe, jezeli rozszerzysz zakres dat.
3. Zmien `unstack("channel")` na `unstack(["hour", "channel"])` i sprawdz ksztalt wyniku.